# E-Commerce Analytics Project
## 02 — Data Quality Analysis

### Objective
Assess the quality, consistency, and reliability of the raw e-commerce datasets before using them for business analysis.

This notebook will focus on:
- Missing values
- Duplicate records
- Data types
- Unexpected categories
- Outliers
- Referential integrity
- Row counts
- Basic financial validation
- Cleaning decisions

The goal is to create clean, validated processed datasets for downstream SQL, Python analysis, and dashboard development.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATABASE_DIR = PROJECT_ROOT / "database"

DATABASE_PATH = DATABASE_DIR / "ecommerce.db"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)
print("Database:", DATABASE_PATH)

Project root: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics
Raw data folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/raw
Processed data folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/processed
Database: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/database/ecommerce.db


In [3]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")
returns = pd.read_csv(RAW_DIR / "returns.csv")
marketing = pd.read_csv(RAW_DIR / "marketing.csv")

In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "returns": returns,
    "marketing": marketing
}

In [5]:
for name, df in datasets.items():
    print(
        f"{name:<12} "
        f"{df.shape[0]:>8,} rows | "
        f"{df.shape[1]} columns"
    )

customers      25,025 rows | 7 columns
products          150 rows | 7 columns
orders         75,000 rows | 7 columns
order_items   125,273 rows | 6 columns
returns        10,269 rows | 5 columns
marketing       6,576 rows | 7 columns


In [6]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


CUSTOMERS
customer_id            object
signup_date            object
city                   object
state                  object
region                 object
age_group              object
acquisition_channel    object
dtype: object

PRODUCTS
product_id            object
product_name          object
category              object
subcategory           object
unit_cost            float64
list_price           float64
popularity_weight    float64
dtype: object

ORDERS
order_id          object
customer_id       object
order_date        object
device            object
sales_channel     object
shipping_cost    float64
order_status      object
dtype: object

ORDER_ITEMS
order_item_id     object
order_id          object
product_id        object
quantity           int64
unit_price       float64
discount_pct     float64
dtype: object

RETURNS
return_id         object
order_item_id     object
return_date       object
return_reason     object
refund_amount    float64
dtype: object

MARKETING
marke

In [7]:
for name, df in datasets.items():
    print(f"\n{name.upper()} NULL VALUES")
    print(df.isnull().sum())


CUSTOMERS NULL VALUES
customer_id              0
signup_date              0
city                   251
state                    0
region                   0
age_group              125
acquisition_channel      0
dtype: int64

PRODUCTS NULL VALUES
product_id           0
product_name         0
category             0
subcategory          0
unit_cost            0
list_price           0
popularity_weight    0
dtype: int64

ORDERS NULL VALUES
order_id           0
customer_id        0
order_date         0
device           375
sales_channel      0
shipping_cost      0
order_status       0
dtype: int64

ORDER_ITEMS NULL VALUES
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
discount_pct     0
dtype: int64

RETURNS NULL VALUES
return_id          0
order_item_id      0
return_date        0
return_reason    308
refund_amount      0
dtype: int64

MARKETING NULL VALUES
marketing_id     0
date             0
channel          0
campaign         0
spend    

## 2. Duplicate Record Assessment

Duplicate records can inflate customer counts, transaction totals, and other KPIs.

Both exact row duplicates and duplicate primary keys will be evaluated before determining the appropriate cleaning action.

In [8]:
duplicate_summary = []

for name, df in datasets.items():

    duplicate_summary.append({
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(
    duplicate_summary
)

duplicate_summary

,dataset,rows,exact_duplicate_rows
0,customers,25025,25
1,products,150,0
2,orders,75000,0
3,order_items,125273,0
4,returns,10269,0
5,marketing,6576,0


In [9]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "returns": "return_id",
    "marketing": "marketing_id"
}

for name, key in primary_keys.items():

    duplicate_keys = (
        datasets[name][key]
        .duplicated()
        .sum()
    )

    print(
        f"{name:<12} | "
        f"duplicate {key}: "
        f"{duplicate_keys:,}"
    )

customers    | duplicate customer_id: 25
products     | duplicate product_id: 0
orders       | duplicate order_id: 0
order_items  | duplicate order_item_id: 0
returns      | duplicate return_id: 0
marketing    | duplicate marketing_id: 0


## 3. Categorical Consistency

Categorical fields will be reviewed for inconsistent spelling, capitalization, unexpected values, and duplicated business categories.

Inconsistent categories can fragment aggregations and produce incorrect dashboard results.

In [10]:
products["category"].value_counts(
    dropna=False
)

category
Beauty & Health      34
Home & Kitchen       30
Sports & Outdoors    28
Electronics          27
Fashion              24
Home and Kitchen      7
Name: count, dtype: int64

In [11]:
customers["acquisition_channel"].value_counts(
    dropna=False
)

acquisition_channel
Email             4253
Organic Search    4190
Direct            4174
Paid Search       4168
Social Media      4161
Affiliate         4079
Name: count, dtype: int64

In [12]:
orders["device"].value_counts(
    dropna=False
)

device
Mobile     43234
Desktop    26963
Tablet      4428
NaN          375
Name: count, dtype: int64

In [13]:
orders["order_status"].value_counts(
    dropna=False
)

order_status
Completed    72756
Cancelled     2244
Name: count, dtype: int64

In [14]:
returns["return_reason"].value_counts(
    dropna=False
)

return_reason
Wrong Item         1697
Wrong Size         1673
Damaged            1666
Changed Mind       1663
Not as Expected    1644
Late Delivery      1618
NaN                 308
Name: count, dtype: int64

In [15]:
marketing["channel"].value_counts(
    dropna=False
)

channel
Paid Search       1096
Email             1096
Affiliate         1096
Display Ads       1096
Organic Search    1096
Social Media       986
social media       110
Name: count, dtype: int64

## 4. Outlier Assessment

Numeric fields will be reviewed for unusual, extreme, or potentially invalid values.

Outliers will not be removed automatically. Each unusual value will first be evaluated to determine whether it represents:
- A valid business observation
- A data-entry problem
- A system error
- A value requiring further investigation

In [16]:
orders[
    ["shipping_cost"]
].describe()

,shipping_cost
count,75000.000000
mean,4.807072
std,4.419948
min,0.000000
25%,0.000000
50%,4.990000
75%,7.990000
max,99.990000


In [17]:
order_items[
    ["quantity", "unit_price", "discount_pct"]
].describe()

,quantity,unit_price,discount_pct
count,125273.000000,125273.000000,125273.000000
mean,1.383953,173.061765,0.089974
std,0.691902,100.106706,0.104555
min,1.000000,12.560000,0.000000
25%,1.000000,89.050000,0.000000
50%,1.000000,152.000000,0.050000
75%,2.000000,251.160000,0.150000
max,4.000000,375.100000,0.400000


In [18]:
returns[
    ["refund_amount"]
].describe()

,refund_amount
count,10269.000000
mean,206.796873
std,174.117902
min,7.540000
25%,85.830000
50%,168.170000
75%,264.210000
max,1485.280000


In [19]:
marketing[
    ["spend", "impressions", "clicks"]
].describe()

,spend,impressions,clicks
count,6576.000000,6576.000000,6556.000000
mean,703.637987,95071.220651,3462.553234
std,625.712580,49045.370419,2508.423071
min,0.000000,10028.000000,58.000000
25%,171.345000,53043.500000,1407.750000
50%,579.160000,95129.500000,2942.500000
75%,1086.780000,137509.250000,4993.250000
max,2398.230000,179996.000000,14333.000000


In [20]:
orders["shipping_cost"].value_counts().sort_index()

shipping_cost
0.00     26137
4.99     22740
7.99     18804
12.99     7299
99.99       20
Name: count, dtype: int64

In [21]:
shipping_outliers = orders[
    orders["shipping_cost"] > 20
]

print(
    f"Shipping costs above $20: "
    f"{len(shipping_outliers):,}"
)

shipping_outliers.head()

Shipping costs above $20: 20


,order_id,customer_id,order_date,device,sales_channel,shipping_cost,order_status
11774,ORD-011775,CUST-13853,2023-11-25,Mobile,Mobile App,99.99,Completed
12982,ORD-012983,CUST-03851,2024-05-08,Mobile,Website,99.99,Completed
19146,ORD-019147,CUST-02715,2024-07-23,Desktop,Website,99.99,Completed
27071,ORD-027072,CUST-00579,2024-07-18,Mobile,Website,99.99,Completed
28801,ORD-028802,CUST-04651,2025-06-23,Mobile,Website,99.99,Completed


In [22]:
print(
    "Negative shipping costs:",
    (orders["shipping_cost"] < 0).sum()
)

print(
    "Zero or negative quantities:",
    (order_items["quantity"] <= 0).sum()
)

print(
    "Negative prices:",
    (order_items["unit_price"] < 0).sum()
)

print(
    "Discounts below 0%:",
    (order_items["discount_pct"] < 0).sum()
)

print(
    "Discounts above 100%:",
    (order_items["discount_pct"] > 1).sum()
)

print(
    "Negative refunds:",
    (returns["refund_amount"] < 0).sum()
)

print(
    "Negative marketing spend:",
    (marketing["spend"] < 0).sum()
)

print(
    "Clicks greater than impressions:",
    (
        marketing["clicks"]
        > marketing["impressions"]
    ).sum()
)

Negative shipping costs: 0
Zero or negative quantities: 0
Negative prices: 0
Discounts below 0%: 0
Discounts above 100%: 0
Negative refunds: 0
Negative marketing spend: 0
Clicks greater than impressions: 0


## 5. Referential Integrity

Relationships between tables will be validated to ensure that foreign keys reference valid records.

Key relationships:

Customers → Orders  
Orders → Order Items  
Products → Order Items  
Order Items → Returns

In [23]:
invalid_order_customers = (
    ~orders["customer_id"]
    .isin(customers["customer_id"])
).sum()

print(
    "Orders with invalid customer IDs:",
    invalid_order_customers
)

Orders with invalid customer IDs: 0


In [24]:
invalid_item_orders = (
    ~order_items["order_id"]
    .isin(orders["order_id"])
).sum()

print(
    "Order items with invalid order IDs:",
    invalid_item_orders
)

Order items with invalid order IDs: 0


In [25]:
invalid_returns = (
    ~returns["order_item_id"]
    .isin(order_items["order_item_id"])
).sum()

print(
    "Returns with invalid order item IDs:",
    invalid_returns
)

Returns with invalid order item IDs: 0


In [26]:
customers_dates = customers.copy()
orders_dates = orders.copy()
returns_dates = returns.copy()

customers_dates["signup_date"] = pd.to_datetime(
    customers_dates["signup_date"]
)

orders_dates["order_date"] = pd.to_datetime(
    orders_dates["order_date"]
)

returns_dates["return_date"] = pd.to_datetime(
    returns_dates["return_date"]
)

In [27]:
customer_signup = (
    customers_dates[
        ["customer_id", "signup_date"]
    ]
    .drop_duplicates(
        subset="customer_id"
    )
)

In [28]:
order_date_check = orders_dates.merge(
    customer_signup,
    on="customer_id",
    how="left"
)

orders_before_signup = (
    order_date_check["order_date"]
    <
    order_date_check["signup_date"]
).sum()

print(
    "Orders before customer signup:",
    orders_before_signup
)

Orders before customer signup: 0


In [29]:
return_date_check = (
    returns_dates
    .merge(
        order_items[
            ["order_item_id", "order_id"]
        ],
        on="order_item_id",
        how="left"
    )
    .merge(
        orders_dates[
            ["order_id", "order_date"]
        ],
        on="order_id",
        how="left"
    )
)

returns_before_purchase = (
    return_date_check["return_date"]
    <
    return_date_check["order_date"]
).sum()

print(
    "Returns before purchase date:",
    returns_before_purchase
)

Returns before purchase date: 0


## 6. Data Cleaning & Transformation

Based on the data-quality assessment, cleaning actions will now be applied.

The goal is not simply to remove imperfect records. Each issue will be handled according to its business meaning while preserving as much useful information as possible.

Cleaning actions include:
- Removing confirmed duplicate records
- Converting date fields to datetime
- Standardizing inconsistent categories
- Handling missing categorical information
- Flagging missing marketing performance data
- Treating abnormal shipping-cost values
- Removing internal data-generation fields

In [30]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
returns_clean = returns.copy()
marketing_clean = marketing.copy()

In [31]:
customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"]
)

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"]
)

returns_clean["return_date"] = pd.to_datetime(
    returns_clean["return_date"]
)

marketing_clean["date"] = pd.to_datetime(
    marketing_clean["date"]
)

In [32]:
print(customers_clean["signup_date"].dtype)
print(orders_clean["order_date"].dtype)
print(returns_clean["return_date"].dtype)
print(marketing_clean["date"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


In [33]:
print(
    "Customer rows before:",
    len(customers_clean)
)

Customer rows before: 25025


In [34]:
customers_clean = (
    customers_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

In [35]:
print(
    "Customer rows after:",
    len(customers_clean)
)

print(
    "Duplicate customer IDs:",
    customers_clean["customer_id"]
    .duplicated()
    .sum()
)

Customer rows after: 25000
Duplicate customer IDs: 0


In [36]:
customers_clean["city"] = (
    customers_clean["city"]
    .fillna("Unknown")
)

customers_clean["age_group"] = (
    customers_clean["age_group"]
    .fillna("Unknown")
)

In [37]:
customers_clean.isnull().sum()

customer_id            0
signup_date            0
city                   0
state                  0
region                 0
age_group              0
acquisition_channel    0
dtype: int64

In [38]:
products_clean["category"] = (
    products_clean["category"]
    .replace({
        "Home and Kitchen":
        "Home & Kitchen"
    })
)

In [39]:
products_clean["category"].value_counts()

category
Home & Kitchen       37
Beauty & Health      34
Sports & Outdoors    28
Electronics          27
Fashion              24
Name: count, dtype: int64

In [40]:
products_clean = products_clean.drop(
    columns=["popularity_weight"]
)

In [41]:
products_clean.columns

Index(['product_id', 'product_name', 'category', 'subcategory', 'unit_cost',
       'list_price'],
      dtype='object')

In [42]:
orders_clean["device"] = (
    orders_clean["device"]
    .fillna("Unknown")
)

In [43]:
orders_clean["device"].value_counts(
    dropna=False
)

device
Mobile     43234
Desktop    26963
Tablet      4428
Unknown      375
Name: count, dtype: int64

In [44]:
orders_clean["shipping_outlier"] = (
    orders_clean["shipping_cost"] > 20
)

In [45]:
orders_clean[
    orders_clean["shipping_outlier"]
][
    ["order_id", "shipping_cost"]
].head()

,order_id,shipping_cost
11774,ORD-011775,99.99
12982,ORD-012983,99.99
19146,ORD-019147,99.99
27071,ORD-027072,99.99
28801,ORD-028802,99.99


In [46]:
normal_shipping_median = (
    orders_clean.loc[
        ~orders_clean["shipping_outlier"],
        "shipping_cost"
    ]
    .median()
)

normal_shipping_median

4.99

In [47]:
orders_clean.loc[
    orders_clean["shipping_outlier"],
    "shipping_cost"
] = normal_shipping_median

In [48]:
returns_clean["return_reason"] = (
    returns_clean["return_reason"]
    .fillna("Not Recorded")
)

In [49]:
returns_clean[
    "return_reason"
].value_counts(dropna=False)

return_reason
Wrong Item         1697
Wrong Size         1673
Damaged            1666
Changed Mind       1663
Not as Expected    1644
Late Delivery      1618
Not Recorded        308
Name: count, dtype: int64

In [50]:
marketing_clean["channel"] = (
    marketing_clean["channel"]
    .replace({
        "social media":
        "Social Media"
    })
)

In [51]:
marketing_clean[
    "channel"
].value_counts()

channel
Paid Search       1096
Social Media      1096
Email             1096
Affiliate         1096
Display Ads       1096
Organic Search    1096
Name: count, dtype: int64

In [53]:
marketing_clean["clicks_missing"] = (
    marketing_clean["clicks"].isna()
)

In [54]:
marketing_clean[
    "clicks_missing"
].value_counts()

clicks_missing
False    6556
True       20
Name: count, dtype: int64

In [55]:
clean_datasets = {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "returns": returns_clean,
    "marketing": marketing_clean
}

In [56]:
for name, df in clean_datasets.items():

    print(f"\n{name.upper()}")

    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Duplicates:",
        df.duplicated().sum()
    )

    print(
        "Missing values:",
        df.isnull().sum().sum()
    )


CUSTOMERS
Rows: 25,000
Duplicates: 0
Missing values: 0

PRODUCTS
Rows: 150
Duplicates: 0
Missing values: 0

ORDERS
Rows: 75,000
Duplicates: 0
Missing values: 0

ORDER_ITEMS
Rows: 125,273
Duplicates: 0
Missing values: 0

RETURNS
Rows: 10,269
Duplicates: 0
Missing values: 0

MARKETING
Rows: 6,576
Duplicates: 0
Missing values: 20


In [57]:
cleaning_log = pd.DataFrame({
    "dataset": [
        "Customers",
        "Customers",
        "Customers",
        "Products",
        "Products",
        "Orders",
        "Orders",
        "Returns",
        "Marketing",
        "Marketing"
    ],

    "issue": [
        "Duplicate customer records",
        "Missing city",
        "Missing age group",
        "Inconsistent category naming",
        "Internal helper field",
        "Missing device",
        "Abnormal shipping cost",
        "Missing return reason",
        "Inconsistent channel capitalization",
        "Missing clicks"
    ],

    "action": [
        "Removed exact duplicates",
        "Replaced with Unknown",
        "Replaced with Unknown",
        "Standardized to Home & Kitchen",
        "Removed popularity_weight",
        "Replaced with Unknown",
        "Flagged and replaced using median normal shipping cost",
        "Replaced with Not Recorded",
        "Standardized to Social Media",
        "Preserved missing values and added quality flag"
    ]
})

cleaning_log

,dataset,issue,action
0,Customers,Duplicate customer records,Removed exact duplicates
1,Customers,Missing city,Replaced with Unknown
2,Customers,Missing age group,Replaced with Unknown
3,Products,Inconsistent category naming,Standardized to Home & Kitchen
4,Products,Internal helper field,Removed popularity_weight
5,Orders,Missing device,Replaced with Unknown
6,Orders,Abnormal shipping cost,Flagged and replaced using median normal shipp...
7,Returns,Missing return reason,Replaced with Not Recorded
8,Marketing,Inconsistent channel capitalization,Standardized to Social Media
9,Marketing,Missing clicks,Preserved missing values and added quality flag


In [59]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Processed folder:", PROCESSED_DIR)
print("Exists:", PROCESSED_DIR.exists())

Processed folder: /Users/gonzalojenkinsmadrid/E-Commerce-Analytics/data/processed
Exists: True


In [60]:
for name, df in clean_datasets.items():
    file_path = PROCESSED_DIR / f"{name}_clean.csv"
    df.to_csv(file_path, index=False)

print("Processed CSV files saved.")

Processed CSV files saved.


In [61]:
for file in PROCESSED_DIR.iterdir():
    print(file.name)

orders_clean.csv
marketing_clean.csv
customers_clean.csv
order_items_clean.csv
returns_clean.csv
products_clean.csv


In [62]:
mkdir(parents=True, exist_ok=True)

zsh:1: number expected
